----

# Fine-tuning YOLO model on the current Roboflow dataset

After having populated the Roboflow project with some labelled images, we want to update our initial YOLO weights using these images to make the process of labelling even faster. This is exactly the goal of this notebook (should be run on colab for preference to make the training faster).

----

## Dataset
We start by importing the dataset from Roboflow. 

First, we need to install the `roboflow` library (if not done yet) using:
`pip install roboflow`

In [1]:
pip install roboflow

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Now to be able to import our private dataset (GreeningAI), you need to use you API KEY. You can find this key in:

For that, go to: **Roboflow Dashboard** → **Your Workspace** → **Settings** → **API Keys** and copy the **Private API Key**.

For our project:
- The **workspace** name is: `aymane-lahgazi-wcpmp`
- The **project** name is: `greeningai-oqs9y`

In [4]:
from roboflow import Roboflow
import config

# Declaring the workspace and project names
API_KEY = "O0noBubT8pT1bQwMMUSh"

# Connecting to the project
rf = Roboflow(api_key=API_KEY)
project = rf.workspace(config.WORKSPACE).project(config.PROJECT)

loading Roboflow workspace...
loading Roboflow project...


Now that we have connected to the project, let us download the latest version of the dataset.

In [5]:
import os
import shutil
# Delete the older version if it exists
if os.path.exists(config.DATA_ROBOFLOW):      # check if it exists
    shutil.rmtree(config.DATA_ROBOFLOW)       # delete folder and all its contents
    print(f"Deleted: {config.DATA_ROBOFLOW}")
# Fetch all versions metadata
versions = project.versions()

# Extract the latest version number
latest_version_number = max(int(v.version) for v in versions)

print("Latest version:", latest_version_number)

# Download the latest version in your preferred format
dataset = project.version(latest_version_number).download("yolov8")

Latest version: 2


Now the datatset should be correctly downloaded, and you should see it in a folder called: `GreeningAI-version`, where `version`is the latest version of the project. 

Let us rename this folder with a fixed name so that we don't have to look for its name at each time.

In [12]:
# Getting the current path of the dataset
old_path = dataset.location

# rename the folder
os.rename(old_path, config.DATA_ROBOFLOW)

## Fine-tuning YOLO

Now that we have successfully downloaded our dataset, we can now fine-tune our initial YOLO model on this latter so make the process of labelling images faster.

In [13]:
from ultralytics import YOLO
import torch

In [17]:
# Data file name
data_yaml = f"{config.DATA_ROBOFLOW}/data.yaml"

# ======================================================
# 7. AUTOMATIC DEVICE SELECTION
# ======================================================
device = 0 if torch.cuda.is_available() else "cpu"
print("Using device:", device)


# ======================================================
# 8. FINE-TUNE MODEL
# ======================================================
model = YOLO(f"{config.WEIGHTS}/init.pt")

model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=8,
    device=device,
    freeze=[0, 170],                 # freeze early layers
    project="finetuned_models",
    name="refinement_v1",
    save_period=1,
    patience=5,
    lr0=1e-4,                            
    lrf=0.1,                              
    weight_decay=0.0005,
    workers=0,
    exist_ok=False
)

print("🎉 Fine-tuning completed.")


# ======================================================
# 9. SAVE UPDATED LAST WEIGHTS (overwrite allowed)
# ======================================================
shutil.copy("finetuned_models/refinement_v1/weights/last.pt", f"{config.WEIGHTS}/last.pt")

print("💾 Updated weights saved → weights/last.pt")

Using device: cpu
New https://pypi.org/project/ultralytics/8.3.242 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.232 🚀 Python-3.10.19 torch-2.9.1 CPU (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_roboflow/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=[0, 170], half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=weights/init.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=refinement_v12, nbs=64, nms=Fa

divide by zero encountered in matmul
overflow encountered in matmul
invalid value encountered in matmul


       1/50         0G       3.55      3.675      2.394       2404        640: 100% ━━━━━━━━━━━━ 1/1 4.3s/it 4.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.2it/s 0.5s
                   all          2        491      0.444      0.413      0.439       0.18

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


divide by zero encountered in matmul
overflow encountered in matmul
invalid value encountered in matmul


       2/50         0G      3.857      3.536      2.684       3729        640: 100% ━━━━━━━━━━━━ 1/1 6.6s/it 6.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.9it/s 0.5s
                   all          2        491      0.455       0.42      0.447      0.183

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      31/50         0G      1.787      1.196      1.259       3348        640: 100% ━━━━━━━━━━━━ 1/1 4.2s/it 4.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.8it/s 0.4s
                   all          2        491      0.967      0.449      0.471      0.321

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


divide by zero encountered in matmul
overflow encountered in matmul
invalid value encountered in matmul


      32/50         0G      2.026      1.194      1.271       3720        640: 100% ━━━━━━━━━━━━ 1/1 3.9s/it 3.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.8it/s 0.4s
                   all          2        491      0.967      0.449      0.471      0.321

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


divide by zero encountered in matmul
overflow encountered in matmul
invalid value encountered in matmul


      33/50         0G      1.841      1.292      1.227       3961        640: 100% ━━━━━━━━━━━━ 1/1 4.0s/it 4.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.7it/s 0.4s
                   all          2        491      0.967      0.449      0.471      0.321
EarlyStopping: Training stopped early as no improvement observed in last 5 epochs. Best results observed at epoch 28, best model saved as best.pt.
To update EarlyStopping(patience=5) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

33 epochs completed in 0.043 hours.
Optimizer stripped from /Users/aymanelahgazi/Desktop/Other Desktop/personal_projects/GreeningAI/finetuned_models/refinement_v12/weights/last.pt, 52.0MB
Optimizer stripped from /Users/aymanelahgazi/Desktop/Other Desktop/personal_projects/GreeningAI/finetuned_models/refinement_v12/weights/best.pt, 52.0MB

Validating /Users/aymanelahgazi/Desktop/Other Desktop/